# TP5 - Knowledge Distillation

**ENSIAS Deep Learning TP**  
**Topic:** *Knowledge Distillation - De la distillation classique aux methodes avancees*

This notebook is the main deliverable for the TP. It follows the statement in two independent parts:

1. **Part 1:** response-based knowledge distillation on filtered MNIST with a `ResNet-50` teacher, a compact `MicroCNN` student, and an extension with Attention Transfer.
2. **Part 2:** advanced knowledge distillation on filtered CIFAR-10 with a `VGG-16` teacher, a `TinyCNN` student, and a comparison between classical KD, FitNets, and RKD.

The notebook is designed to run in **VS Code** or **Colab** with relative paths only. Long training cells are clearly marked. Figures are saved to `figures/` and tables to `results/`.


## 2. Environment and imports

The heavy logic is intentionally split across `src/` so that the notebook can focus on the TP methodology, experiment configuration, and interpretation.

**Colab note:** upload or clone the whole `tp5_DL` folder, then open this notebook from inside that folder. If you want the full official training runs, switch Colab to **GPU** before executing the notebook. The notebook now detects GPU automatically and enables the long experiments by default when a CUDA runtime is available.


In [1]:
from pathlib import Path
import copy
import json
import math
import os
import sys
import warnings


def detect_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def find_project_root() -> Path:
    env_root = os.environ.get("TP5_PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser().resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "src" / "utils.py").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing src/utils.py. "
        "If you are using Colab, upload or clone the whole tp5_DL folder, then open the notebook from inside that folder."
    )


IN_COLAB = detect_colab()
PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.manifold import TSNE
from sklearn.metrics import precision_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from src.utils import (
    FIGURES_DIR,
    RESULTS_DIR,
    adapt_inputs_to_model,
    build_label_filtered_dataset,
    count_params,
    ensure_project_dirs,
    format_size_kb,
    get_device,
    load_checkpoint_if_available,
    model_size_kb,
    save_checkpoint,
    save_dataframe,
    seed_everything,
    select_indices_by_label,
    summarize_model_table,
)
from src.models import (
    MicroCNN,
    SmallMNISTCNN,
    TinyCNN,
    build_resnet18_student,
    build_resnet50_teacher,
    build_vgg16_teacher,
)
from src.kd_losses import attention_map, at_loss, fitnets_loss, kd_loss, rkd_angle_loss, rkd_distance_loss
from src.train_utils import FeatureHook, evaluate, fit_model, train_epoch, train_supervised_epoch
from src.metrics import class_prototypes, cosine_similarity_matrix, measure_latency
from src.visualization import (
    plot_attention_maps,
    plot_heatmap_grid,
    plot_history_curves,
    plot_soft_label_bars,
    plot_temperature_curve,
    plot_tsne_triptych,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
ensure_project_dirs()
ROOT = PROJECT_ROOT
DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)


## 3. Reproducibility and device setup

The TP fixes the main reproducibility settings before any dataset or model is created.

- On **CPU**, the notebook keeps long training runs disabled by default.
- On a **CUDA GPU runtime** such as Google Colab with GPU enabled, the notebook automatically enables teacher training and the full experiment suite.

You can still override the flags manually in the next cell if you want a faster debug pass or a full official run.


In [2]:
SEED = 42
seed_everything(SEED)
device = get_device()

USE_ACCELERATED_RUNTIME = device.type == "cuda"
RUN_LONG_EXPERIMENTS = USE_ACCELERATED_RUNTIME
TRAIN_TEACHERS = USE_ACCELERATED_RUNTIME
SAVE_INTERMEDIATE_CHECKPOINTS = True
NUM_WORKERS = 2 if USE_ACCELERATED_RUNTIME else 0
PIN_MEMORY = USE_ACCELERATED_RUNTIME

num_classes = 3
batch_size = 32

if IN_COLAB and device.type != "cuda":
    print("Colab detected but CUDA is not active. For full training runs, switch Runtime > Change runtime type > GPU.")
elif device.type == "cuda":
    print("CUDA runtime detected. Full teacher training and long experiments are enabled by default.")
else:
    print("CPU runtime detected. Long experiments stay disabled by default to avoid very slow execution.")

pd.DataFrame(
    {
        "setting": [
            "seed",
            "device",
            "IN_COLAB",
            "RUN_LONG_EXPERIMENTS",
            "TRAIN_TEACHERS",
            "SAVE_INTERMEDIATE_CHECKPOINTS",
            "NUM_WORKERS",
            "PIN_MEMORY",
            "num_classes",
            "batch_size",
        ],
        "value": [
            SEED,
            str(device),
            IN_COLAB,
            RUN_LONG_EXPERIMENTS,
            TRAIN_TEACHERS,
            SAVE_INTERMEDIATE_CHECKPOINTS,
            NUM_WORKERS,
            PIN_MEMORY,
            num_classes,
            batch_size,
        ],
    }
)


,setting,value
0,seed,42
1,device,cpu
2,RUN_LONG_EXPERIMENTS,False
3,TRAIN_TEACHERS,False
4,num_classes,3
5,batch_size,32


# 4. Part 1 - MNIST Response-Based KD

This part follows the statement exactly:

- dataset: `torchvision.datasets.MNIST`
- kept digits: `0`, `1`, `8`
- remapping: `0 -> 0`, `1 -> 1`, `8 -> 2`
- resize: `32 x 32`
- normalization: mean `0.5`, std `0.5`
- batch size: `32`
- teacher: pretrained `ResNet-50` with a 3-class head
- student: `MicroCNN` with exactly three `Conv -> BN -> ReLU` blocks and one fully connected classifier


### P1.1 and P1.2 - Dataset loading

The MNIST pipeline keeps a single grayscale channel for the student. The teacher still uses an ImageNet `ResNet-50`, so the notebook adapts the input to three channels only at model-forward time when needed. This keeps the data pipeline simple and respects the normalization requested in the statement.


In [3]:
mnist_transform = transforms.Compose(
    [
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ]
)

mnist_train_full = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=mnist_transform)
mnist_test_full = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=mnist_transform)

mnist_keep = [0, 1, 8]
mnist_label_map = {0: 0, 1: 1, 8: 2}
mnist_class_names = ["0", "1", "8"]

mnist_train = build_label_filtered_dataset(mnist_train_full, keep_labels=mnist_keep, label_map=mnist_label_map)
mnist_test = build_label_filtered_dataset(mnist_test_full, keep_labels=mnist_keep, label_map=mnist_label_map)

mnist_train_loader = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
mnist_test_loader = DataLoader(mnist_test, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

mnist_counts = pd.DataFrame(
    {
        "split": ["train"] * len(mnist_train) + ["test"] * len(mnist_test),
        "label": [label for _, label in mnist_train] + [label for _, label in mnist_test],
    }
).groupby(["split", "label"]).size().reset_index(name="count")
mnist_counts["class_name"] = mnist_counts["label"].map(dict(enumerate(mnist_class_names)))
mnist_counts


100.0%
100.0%
100.0%
100.0%


,split,label,count,class_name
0,test,0,980,0
1,test,1,1135,1
2,test,2,974,8
3,train,0,5923,0
4,train,1,6742,1
5,train,2,5851,8


### P1.3 - Models and teacher preparation

The teacher is built from torchvision `ResNet-50` weights and then its classification head is replaced with `3` outputs. Distillation requires a trained teacher on the filtered MNIST task, so the notebook first tries to load a checkpoint from `results/`. If none exists, the long training cell below can be enabled.

The `MicroCNN` student has exactly three `Conv -> BN -> ReLU` blocks and stays under `100,000` parameters.


In [4]:
mnist_teacher_bundle = build_resnet50_teacher(num_classes=num_classes)
mnist_teacher = mnist_teacher_bundle.model.to(device)
mnist_teacher_ckpt = RESULTS_DIR / "mnist_teacher_resnet50.pt"
mnist_teacher_ready = load_checkpoint_if_available(mnist_teacher, mnist_teacher_ckpt, map_location=device)
mnist_teacher.eval()

micro_student_template = MicroCNN(num_classes=num_classes).to(device)
small_student_template = SmallMNISTCNN(num_classes=num_classes).to(device)
large_student_template = build_resnet18_student(num_classes=num_classes).model.to(device)

assert count_params(micro_student_template, verbose=False) < 100_000, "MicroCNN exceeds 100k parameters."
assert count_params(small_student_template, verbose=False) < 10_000, "Small student exceeds 10k parameters."

print(mnist_teacher_bundle.message)
print(f"Teacher checkpoint found: {mnist_teacher_ready}")
print(f"MicroCNN parameters: {count_params(micro_student_template, verbose=False)}")
print(f"Small student parameters: {count_params(small_student_template, verbose=False)}")


Loaded ResNet-50 with ImageNet pretrained weights.
Teacher checkpoint found: False
MicroCNN parameters: 21075
Small student parameters: 3283


### Long-run cell - optional MNIST teacher fine-tuning

This cell is intentionally separated because distillation should not start from a random teacher head. If `TRAIN_TEACHERS = False`, the notebook only prepares the function and prints a reminder.


In [5]:
def train_mnist_teacher(epochs: int = 5, lr: float = 1e-4):
    teacher_result = build_resnet50_teacher(num_classes=num_classes)
    teacher_model = teacher_result.model.to(device)
    optimizer = optim.Adam(teacher_model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        train_metrics = train_supervised_epoch(teacher_model, mnist_train_loader, optimizer, device)
        test_metrics = evaluate(teacher_model, mnist_test_loader, device)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_acc": train_metrics["acc"],
                "test_loss": test_metrics["loss"],
                "test_acc": test_metrics["acc"],
            }
        )
        print(f"[MNIST teacher] epoch={epoch:02d} train_acc={train_metrics['acc']:.4f} test_acc={test_metrics['acc']:.4f}")
    history_df = pd.DataFrame(history)
    if SAVE_INTERMEDIATE_CHECKPOINTS:
        save_checkpoint(teacher_model, mnist_teacher_ckpt)
    teacher_model.eval()
    return teacher_model, history_df

if mnist_teacher_ready:
    print(f"Loaded MNIST teacher checkpoint from {mnist_teacher_ckpt}.")
elif TRAIN_TEACHERS:
    mnist_teacher, mnist_teacher_history = train_mnist_teacher()
    mnist_teacher_ready = True
    mnist_teacher.eval()
    display(mnist_teacher_history.tail())
else:
    print("No MNIST teacher checkpoint found. Set TRAIN_TEACHERS=True to fine-tune the teacher before KD experiments.")


No MNIST teacher checkpoint found. Set TRAIN_TEACHERS=True to fine-tune the teacher before KD experiments.


### P1.4 - Model statistics

The TP asks for the number of parameters and the approximate model size. The helper functions `count_params(model)` and `model_size_kb(model)` are reused here for both teacher and student models.


In [6]:
part1_model_stats = summarize_model_table(
    [
        {
            "model": "Teacher - ResNet50",
            "params": count_params(mnist_teacher, verbose=False),
            "params_k": count_params(mnist_teacher, verbose=False) / 1_000,
            "approx_size_kb": model_size_kb(mnist_teacher),
            "approx_size": format_size_kb(model_size_kb(mnist_teacher)),
        },
        {
            "model": "Student - MicroCNN",
            "params": count_params(micro_student_template, verbose=False),
            "params_k": count_params(micro_student_template, verbose=False) / 1_000,
            "approx_size_kb": model_size_kb(micro_student_template),
            "approx_size": format_size_kb(model_size_kb(micro_student_template)),
        },
    ],
    filename="part1_model_stats.csv",
)
part1_model_stats


,model,params,params_k,approx_size_kb,approx_size
0,Teacher - ResNet50,23514179,23514.179,92060.175781,89.90 MB
1,Student - MicroCNN,21075,21.075,83.097656,83.10 KB


### P1.5 - Soft-label visualization on ambiguous digit 8 samples

The goal is to inspect whether temperature reveals teacher hesitation between the remapped classes `0` and `8`. The cell below selects five digit-`8` samples with the smallest teacher margin between class `8` and class `0`, then plots teacher probabilities for `T=1` and `T=4`.

If no trained teacher is available yet, the cell exits cleanly instead of inventing observations.


In [7]:
@torch.no_grad()
def teacher_probabilities(model, image_tensor, T: float = 1.0):
    logits = model(adapt_inputs_to_model(model, image_tensor.unsqueeze(0).to(device)))
    return torch.softmax(logits / T, dim=1).squeeze(0).cpu()

@torch.no_grad()
def select_ambiguous_digit_eights(model, dataset, top_k: int = 5):
    scored = []
    for index, (image, label) in enumerate(dataset):
        if label != 2:
            continue
        probs = teacher_probabilities(model, image, T=1.0)
        margin = float(abs(probs[2] - probs[0]))
        scored.append((margin, index, image.cpu(), probs))
    scored.sort(key=lambda item: item[0])
    return scored[:top_k]

if mnist_teacher_ready:
    ambiguous_eights = select_ambiguous_digit_eights(mnist_teacher, mnist_test, top_k=5)
    amb_images = np.stack([item[2].squeeze(0).numpy() for item in ambiguous_eights])
    amb_probs_t1 = np.stack([teacher_probabilities(mnist_teacher, item[2], T=1.0).numpy() for item in ambiguous_eights])
    amb_probs_t4 = np.stack([teacher_probabilities(mnist_teacher, item[2], T=4.0).numpy() for item in ambiguous_eights])
    plot_soft_label_bars(
        amb_images,
        amb_probs_t1,
        amb_probs_t4,
        class_names=mnist_class_names,
        title="MNIST soft labels on visually ambiguous digit 8 samples",
        filename="part1_soft_labels_ambiguous_8.png",
    )
    plt.show()
else:
    print("Soft-label visualization is pending because the MNIST teacher checkpoint is not available yet.")


Soft-label visualization is pending because the MNIST teacher checkpoint is not available yet.


When the previous figure is executed with a trained teacher, the expected qualitative check is simple: **if `T=4` makes the bar chart flatter than `T=1`, then teacher uncertainty becomes more visible**. Keep the final written observation tied to the actual plots shown above rather than to a guessed outcome.


### P1.6 - Manual KD loss verification

For the toy logits

- `zT = [1.2, 0.5, 4.1]`
- `zS = [0.9, 0.4, 3.6]`
- `y = 2`
- `T = 4`
- `alpha = 0.4`

we use the notebook convention implemented in `src.kd_losses.kd_loss`:

\[
\mathcal{L}_{KD} = lpha \cdot CE(z_S, y) + (1-lpha) \cdot \left[-
rac{1}{C}\sum_{c=1}^{C} p_T^{(c)}(T) \log p_S^{(c)}(T)
ight]
\]

with `C = 3` classes. Using rounded softened probabilities

- `pT(T=4) pprox [0.26, 0.21, 0.53]`
- `pS(T=4) pprox [0.26, 0.23, 0.51]`

and `CE(zS, y) pprox 0.103`, the class-mean soft term is about `0.32`, hence

\[
\mathcal{L}_{KD} pprox 0.4 	imes 0.103 + 0.6 	imes 0.32 pprox 0.23
\]

The tensor implementation below computes the exact value from the logits without manual rounding.


In [ ]:
toy_teacher = torch.tensor([[1.2, 0.5, 4.1]], dtype=torch.float32)
toy_student = torch.tensor([[0.9, 0.4, 3.6]], dtype=torch.float32)
toy_labels = torch.tensor([2])
manual_kd_value = kd_loss(toy_student, toy_teacher, toy_labels, T=4.0, alpha=0.4)
print(f"Exact kd_loss value from the implementation: {manual_kd_value.item():.6f}")


### P1.7 - Training and evaluation loop

The reusable training loop is implemented in `src.train_utils`:

- `train_epoch(student, teacher, loader, optimizer, T, alpha)` for response-based KD
- `evaluate(model, loader)` for supervised evaluation

The teacher forward pass is already wrapped in `torch.no_grad()` inside the helper.


In [ ]:
part1_runs = {}

@torch.no_grad()
def evaluate_macro_precision(model, loader):
    model.eval()
    predictions = []
    targets = []
    for inputs, labels in loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        logits = model(adapt_inputs_to_model(model, inputs))
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
        targets.extend(labels.cpu().tolist())
    return precision_score(targets, predictions, average="macro", zero_division=0)


def deployment_flag(size_kb: float, latency_ms: float, accuracy: float, size_limit_kb: float, latency_limit_ms: float) -> str:
    if np.isnan(accuracy):
        return "Pending run"
    return "Yes" if size_kb <= size_limit_kb and latency_ms <= latency_limit_ms else "No"


def run_experiment(T: float, alpha: float, label: str, epochs: int):
    student = MicroCNN(num_classes=num_classes).to(device)
    optimizer = optim.Adam(student.parameters(), lr=1e-3)
    if alpha < 1.0 and not mnist_teacher_ready:
        raise RuntimeError("The MNIST KD experiment requires a trained teacher checkpoint.")

    history = fit_model(
        student=student,
        train_loader=mnist_train_loader,
        test_loader=mnist_test_loader,
        device=device,
        epochs=epochs,
        optimizer=optimizer,
        teacher=None if alpha >= 1.0 else mnist_teacher,
        T=T,
        alpha=alpha,
    )
    precision = evaluate_macro_precision(student, mnist_test_loader)
    latency = measure_latency(student, input_shape=(1, 1, 32, 32), device=device, n=100, warmup=10)
    summary = {
        "model": label,
        "size_kb": model_size_kb(student),
        "latency_ms": latency,
        "precision": precision,
        "IoT chip OK?": deployment_flag(model_size_kb(student), latency, history["test_acc"].iloc[-1], 1024.0, 10.0),
    }
    return {"student": student, "history": history, "summary": summary}


### P1.8 - Experiment 1: student alone vs student with KD

The statement asks for 30 epochs with Adam (`lr=1e-3`) for:

1. `Student alone`: `T = 1`, `alpha = 1.0`
2. `Student + KD`: `T = 4`, `alpha = 0.4`

The cell below is intentionally long-run. It also saves the summary table to `results/part1_experiment1_summary.csv`.


In [ ]:
if RUN_LONG_EXPERIMENTS:
    part1_runs["student_alone"] = run_experiment(T=1.0, alpha=1.0, label="Student alone", epochs=30)
    part1_runs["student_kd"] = run_experiment(T=4.0, alpha=0.4, label="Student + KD", epochs=30)

    plot_history_curves(
        {
            "Student alone": part1_runs["student_alone"]["history"],
            "Student + KD": part1_runs["student_kd"]["history"],
        },
        filename="part1_experiment1_accuracy_curves.png",
    )
    plt.show()

    experiment1_summary = pd.DataFrame([run["summary"] for run in part1_runs.values()])
    experiment1_summary["size"] = experiment1_summary["size_kb"].map(format_size_kb)
    save_dataframe(experiment1_summary, "part1_experiment1_summary.csv")
    display(experiment1_summary[["model", "size", "latency_ms", "precision", "IoT chip OK?"]])
else:
    print("Experiment 1 is prepared but not executed. Set RUN_LONG_EXPERIMENTS=True to launch the 30-epoch runs.")


### P1.9 - Experiment 2: temperature effect

This sweep fixes `alpha = 0.4`, trains for `15` epochs, and compares `T in {1, 2, 4, 8}`. The observation column is left as an interpretation prompt so that the final report stays tied to the measured results.


In [ ]:
if RUN_LONG_EXPERIMENTS:
    temperature_rows = []
    for temperature in [1.0, 2.0, 4.0, 8.0]:
        run = run_experiment(T=temperature, alpha=0.4, label=f"KD @ T={temperature}", epochs=15)
        temperature_rows.append(
            {
                "temperature": temperature,
                "test_acc": run["history"]["test_acc"].iloc[-1],
                "observation": "Check whether the softer distribution helps separate ambiguous 0/8 samples.",
            }
        )
    temperature_df = pd.DataFrame(temperature_rows)
    save_dataframe(temperature_df, "part1_temperature_sweep.csv")
    display(temperature_df)
    plot_temperature_curve(temperature_df, filename="part1_temperature_effect.png")
    plt.show()
else:
    print("Temperature sweep prepared. Enable RUN_LONG_EXPERIMENTS to run the 15-epoch experiments.")


### P1.10 - Experiment 3: three distillation regimes

The three requested students are:

1. **Large student:** `ResNet-18` with a 3-class head
2. **Medium student:** `MicroCNN`
3. **Small student:** a two-block CNN with fewer than `10,000` parameters

Each student is trained **with and without KD** for `15` epochs using `T=4`, `alpha=0.4`.


In [ ]:
def build_part1_student(regime: str):
    if regime == "Large Student":
        return build_resnet18_student(num_classes=num_classes).model.to(device)
    if regime == "Medium Student":
        return MicroCNN(num_classes=num_classes).to(device)
    if regime == "Small Student":
        return SmallMNISTCNN(num_classes=num_classes).to(device)
    raise ValueError(f"Unknown regime: {regime}")

if RUN_LONG_EXPERIMENTS:
    regime_rows = []
    for regime in ["Large Student", "Medium Student", "Small Student"]:
        supervised_student = build_part1_student(regime)
        supervised_history = fit_model(
            student=supervised_student,
            train_loader=mnist_train_loader,
            test_loader=mnist_test_loader,
            device=device,
            epochs=15,
            optimizer=optim.Adam(supervised_student.parameters(), lr=1e-3),
            teacher=None,
        )

        kd_student = build_part1_student(regime)
        kd_history = fit_model(
            student=kd_student,
            train_loader=mnist_train_loader,
            test_loader=mnist_test_loader,
            device=device,
            epochs=15,
            optimizer=optim.Adam(kd_student.parameters(), lr=1e-3),
            teacher=mnist_teacher,
            T=4.0,
            alpha=0.4,
        )

        regime_rows.append(
            {
                "student": regime,
                "params": count_params(kd_student, verbose=False),
                "accuracy without KD": supervised_history["test_acc"].iloc[-1],
                "accuracy with KD": kd_history["test_acc"].iloc[-1],
                "observed regime/comment": "Fill with the measured gain or saturation pattern after execution.",
            }
        )
    regime_df = pd.DataFrame(regime_rows)
    save_dataframe(regime_df, "part1_regime_comparison.csv")
    display(regime_df)
else:
    print("Regime comparison prepared. It requires the MNIST teacher checkpoint and long execution time.")


### P1.11 - Attention Transfer (AT)

Attention Transfer uses the channel-wise mean of squared activations. The hooks below compare:

- teacher feature map: `ResNet50.layer2`
- student feature map: `MicroCNN.block3`

This choice keeps the spatial resolutions compatible on `32 x 32` inputs.


In [ ]:
def run_attention_transfer_experiment(epochs: int = 30, T: float = 4.0, alpha: float = 0.4, beta: float = 0.1):
    if not mnist_teacher_ready:
        raise RuntimeError("Attention Transfer requires a trained MNIST teacher checkpoint.")

    student = MicroCNN(num_classes=num_classes).to(device)
    student_before = copy.deepcopy(student).to(device)
    optimizer = optim.Adam(student.parameters(), lr=1e-3)

    teacher_hook = FeatureHook(mnist_teacher.layer2)
    student_hook = FeatureHook(student.block3)

    def extra_loss_fn(base_loss, student_logits, teacher_logits):
        return base_loss + at_loss(student_hook.output, teacher_hook.output, beta=beta)

    history = fit_model(
        student=student,
        train_loader=mnist_train_loader,
        test_loader=mnist_test_loader,
        device=device,
        epochs=epochs,
        optimizer=optimizer,
        teacher=mnist_teacher,
        T=T,
        alpha=alpha,
        extra_loss_fn=extra_loss_fn,
    )

    teacher_hook.close()
    student_hook.close()
    return student_before, student, history

@torch.no_grad()
def get_attention_heatmap(model, hook_module, image_tensor):
    hook = FeatureHook(hook_module)
    _ = model(adapt_inputs_to_model(model, image_tensor.unsqueeze(0).to(device)))
    feature_map = hook.output
    hook.close()
    return attention_map(feature_map).squeeze(0).cpu().numpy()

if RUN_LONG_EXPERIMENTS:
    student_before_at, student_after_at, at_history = run_attention_transfer_experiment()
    part1_runs["student_kd_at"] = {
        "student": student_after_at,
        "history": at_history,
        "summary": {
            "model": "Student + KD + AT",
            "size_kb": model_size_kb(student_after_at),
            "latency_ms": measure_latency(student_after_at, input_shape=(1, 1, 32, 32), device=device),
            "precision": evaluate_macro_precision(student_after_at, mnist_test_loader),
        },
    }
    if mnist_teacher_ready:
        sample_image, _ = mnist_test[select_indices_by_label(mnist_test, per_label=1, num_classes=3)[-1]]
        plot_attention_maps(
            image=sample_image.squeeze(0).numpy(),
            teacher_map=get_attention_heatmap(mnist_teacher, mnist_teacher.layer2, sample_image),
            student_before=get_attention_heatmap(student_before_at, student_before_at.block3, sample_image),
            student_after=get_attention_heatmap(student_after_at, student_after_at.block3, sample_image),
            filename="part1_attention_maps_before_after.png",
        )
        plt.show()
else:
    print("Attention Transfer pipeline prepared. Enable RUN_LONG_EXPERIMENTS to train the KD + AT student.")


### P1.12 - Latency and final Part 1 synthesis

Latency is measured with `10` warm-up passes and `100` timed passes on an input shape `1 x 1 x 32 x 32`. The table below is populated once the corresponding experiments have been executed.


In [ ]:
part1_synthesis_rows = [
    {
        "model": "Teacher",
        "size": format_size_kb(model_size_kb(mnist_teacher)),
        "latency_ms": measure_latency(mnist_teacher, input_shape=(1, 1, 32, 32), device=device),
        "precision": evaluate_macro_precision(mnist_teacher, mnist_test_loader) if mnist_teacher_ready else np.nan,
        "IoT chip OK?": deployment_flag(model_size_kb(mnist_teacher), measure_latency(mnist_teacher, (1, 1, 32, 32), device), np.nan if not mnist_teacher_ready else 1.0, 1024.0, 10.0),
    }
]

for key, display_name in [
    ("student_alone", "Student alone"),
    ("student_kd", "Student + KD"),
    ("student_kd_at", "Student + KD + AT"),
]:
    if key in part1_runs:
        run = part1_runs[key]
        part1_synthesis_rows.append(
            {
                "model": display_name,
                "size": format_size_kb(run["summary"]["size_kb"]),
                "latency_ms": run["summary"]["latency_ms"],
                "precision": run["summary"]["precision"],
                "IoT chip OK?": deployment_flag(run["summary"]["size_kb"], run["summary"]["latency_ms"], run["summary"]["precision"], 1024.0, 10.0),
            }
        )

part1_synthesis = pd.DataFrame(part1_synthesis_rows)
save_dataframe(part1_synthesis, "part1_final_synthesis.csv")
part1_synthesis


# 5. Part 2 - CIFAR-10 Advanced KD

This second part keeps only the CIFAR-10 classes `cat(3)`, `deer(4)`, `dog(5)`, and `horse(7)`, remapped to labels `0..3`. The student is a compact `TinyCNN`, while the teacher is a `VGG-16` with a 4-class head.


### P2.1 - Dataset

The training transform adds random horizontal flipping as requested. The normalization constants follow the CIFAR-10 statistics from the statement.


In [ ]:
part2_num_classes = 4
part2_batch_size = 64
cifar_class_names = ["cat", "dog", "deer", "horse"]
cifar_keep = [3, 5, 4, 7]
cifar_label_map = {3: 0, 5: 1, 4: 2, 7: 3}

cifar_train_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]
)
cifar_test_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]
)

cifar_train_full = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=cifar_train_transform)
cifar_test_full = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=cifar_test_transform)

cifar_train = build_label_filtered_dataset(cifar_train_full, keep_labels=cifar_keep, label_map=cifar_label_map)
cifar_test = build_label_filtered_dataset(cifar_test_full, keep_labels=cifar_keep, label_map=cifar_label_map)

cifar_train_loader = DataLoader(cifar_train, batch_size=part2_batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
cifar_test_loader = DataLoader(cifar_test, batch_size=part2_batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

cifar_counts = pd.DataFrame(
    {
        "split": ["train"] * len(cifar_train) + ["test"] * len(cifar_test),
        "label": [label for _, label in cifar_train] + [label for _, label in cifar_test],
    }
).groupby(["split", "label"]).size().reset_index(name="count")
cifar_counts["class_name"] = cifar_counts["label"].map(dict(enumerate(cifar_class_names)))
cifar_counts


### P2.2 - Models and teacher preparation

The `TinyCNN` student follows the requested architecture exactly:

- conv channels: `16 -> 32 -> 64 -> 128`
- `BatchNorm + ReLU` after each convolution
- max-pooling after layers `2` and `4`
- one final fully connected layer with `4` outputs
- total parameter count under `200,000`


In [ ]:
vgg_teacher_bundle = build_vgg16_teacher(num_classes=part2_num_classes)
vgg_teacher = vgg_teacher_bundle.model.to(device)
vgg_teacher_ckpt = RESULTS_DIR / "cifar_teacher_vgg16.pt"
vgg_teacher_ready = load_checkpoint_if_available(vgg_teacher, vgg_teacher_ckpt, map_location=device)
vgg_teacher.eval()

tiny_student_template = TinyCNN(num_classes=part2_num_classes).to(device)
assert count_params(tiny_student_template, verbose=False) < 200_000, "TinyCNN exceeds 200k parameters."

print(vgg_teacher_bundle.message)
print(f"Teacher checkpoint found: {vgg_teacher_ready}")
print(f"TinyCNN parameters: {count_params(tiny_student_template, verbose=False)}")

part2_model_stats = summarize_model_table(
    [
        {
            "model": "Teacher - VGG16",
            "params": count_params(vgg_teacher, verbose=False),
            "params_k": count_params(vgg_teacher, verbose=False) / 1_000,
            "approx_size_kb": model_size_kb(vgg_teacher),
            "approx_size": format_size_kb(model_size_kb(vgg_teacher)),
        },
        {
            "model": "Student - TinyCNN",
            "params": count_params(tiny_student_template, verbose=False),
            "params_k": count_params(tiny_student_template, verbose=False) / 1_000,
            "approx_size_kb": model_size_kb(tiny_student_template),
            "approx_size": format_size_kb(model_size_kb(tiny_student_template)),
        },
    ],
    filename="part2_model_stats.csv",
)
part2_model_stats


### Long-run cell - optional CIFAR-10 teacher fine-tuning

As in Part 1, the distillation runs should not start from an untrained teacher head. The cell below fine-tunes the `VGG-16` teacher only when `TRAIN_TEACHERS = True`.


In [ ]:
def train_cifar_teacher(epochs: int = 5, lr: float = 1e-4):
    teacher_result = build_vgg16_teacher(num_classes=part2_num_classes)
    teacher_model = teacher_result.model.to(device)
    optimizer = optim.Adam(teacher_model.parameters(), lr=lr)
    history = []
    for epoch in range(1, epochs + 1):
        train_metrics = train_supervised_epoch(teacher_model, cifar_train_loader, optimizer, device)
        test_metrics = evaluate(teacher_model, cifar_test_loader, device)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_acc": train_metrics["acc"],
                "test_loss": test_metrics["loss"],
                "test_acc": test_metrics["acc"],
            }
        )
        print(f"[CIFAR teacher] epoch={epoch:02d} train_acc={train_metrics['acc']:.4f} test_acc={test_metrics['acc']:.4f}")
    history_df = pd.DataFrame(history)
    if SAVE_INTERMEDIATE_CHECKPOINTS:
        save_checkpoint(teacher_model, vgg_teacher_ckpt)
    teacher_model.eval()
    return teacher_model, history_df

if vgg_teacher_ready:
    print(f"Loaded CIFAR teacher checkpoint from {vgg_teacher_ckpt}.")
elif TRAIN_TEACHERS:
    vgg_teacher, vgg_teacher_history = train_cifar_teacher()
    vgg_teacher_ready = True
    vgg_teacher.eval()
    display(vgg_teacher_history.tail())
else:
    print("No CIFAR teacher checkpoint found. Set TRAIN_TEACHERS=True before running the advanced KD experiments.")


### P2.3 - Baseline reference: student alone vs classical KD

These runs provide the common reference for the advanced methods. The notebook reuses `kd_loss` from Part 1.


In [ ]:
part2_runs = {}

if RUN_LONG_EXPERIMENTS:
    if not vgg_teacher_ready:
        raise RuntimeError("Baseline KD on CIFAR-10 requires a trained VGG-16 teacher checkpoint.")

    baseline_student = TinyCNN(num_classes=part2_num_classes).to(device)
    baseline_history = fit_model(
        student=baseline_student,
        train_loader=cifar_train_loader,
        test_loader=cifar_test_loader,
        device=device,
        epochs=30,
        optimizer=optim.Adam(baseline_student.parameters(), lr=1e-3),
        teacher=None,
    )
    part2_runs["student_alone"] = {"student": baseline_student, "history": baseline_history}

    kd_student = TinyCNN(num_classes=part2_num_classes).to(device)
    kd_history = fit_model(
        student=kd_student,
        train_loader=cifar_train_loader,
        test_loader=cifar_test_loader,
        device=device,
        epochs=30,
        optimizer=optim.Adam(kd_student.parameters(), lr=1e-3),
        teacher=vgg_teacher,
        T=4.0,
        alpha=0.4,
    )
    part2_runs["student_kd"] = {"student": kd_student, "history": kd_history}

    baseline_reference = pd.DataFrame(
        [
            {"model": "Student alone", "test_acc": baseline_history["test_acc"].iloc[-1]},
            {"model": "Student + KD", "test_acc": kd_history["test_acc"].iloc[-1]},
        ]
    )
    save_dataframe(baseline_reference, "part2_baseline_reference.csv")
    display(baseline_reference)
else:
    print("Part 2 baseline runs are prepared but not executed.")


### P2.4 - FitNets

We supervise an intermediate feature pair with hooks:

- teacher feature: `VGG16.features[8]`
- student feature: `TinyCNN.pool1`

Their channel counts differ (`128` vs `32`), so a `1x1` adaptation convolution is added as required.


In [ ]:
def mean_heatmap(feature_tensor: torch.Tensor) -> np.ndarray:
    return feature_tensor.detach().float().pow(2).mean(dim=1).squeeze(0).cpu().numpy()

if RUN_LONG_EXPERIMENTS:
    if not vgg_teacher_ready:
        raise RuntimeError("FitNets requires a trained VGG-16 teacher checkpoint.")

    fitnets_rows = []
    fitnets_visuals = {}
    for gamma in [0.1, 0.5, 1.0]:
        student = TinyCNN(num_classes=part2_num_classes).to(device)
        adapter = nn.Conv2d(32, 128, kernel_size=1).to(device)
        optimizer = optim.Adam(list(student.parameters()) + list(adapter.parameters()), lr=1e-3)
        teacher_hook = FeatureHook(vgg_teacher.features[8])
        student_hook = FeatureHook(student.pool1)

        def extra_loss_fn(base_loss, student_logits, teacher_logits):
            return fitnets_loss(student_hook.output, teacher_hook.output, adapter=adapter, gamma=gamma, kd_loss_val=base_loss)

        history = fit_model(
            student=student,
            train_loader=cifar_train_loader,
            test_loader=cifar_test_loader,
            device=device,
            epochs=30,
            optimizer=optimizer,
            teacher=vgg_teacher,
            T=4.0,
            alpha=0.4,
            extra_loss_fn=extra_loss_fn,
        )
        fitnets_rows.append(
            {
                "gamma": gamma,
                "accuracy": history["test_acc"].iloc[-1],
                "observation": "Compare the feature-alignment gain against the classical KD reference.",
            }
        )
        fitnets_visuals[gamma] = {"student": student, "adapter": adapter}
        teacher_hook.close()
        student_hook.close()

    fitnets_df = pd.DataFrame(fitnets_rows)
    save_dataframe(fitnets_df, "part2_fitnets_results.csv")
    display(fitnets_df)

    cat_index = next(index for index, (_, label) in enumerate(cifar_test) if label == 0)
    dog_index = next(index for index, (_, label) in enumerate(cifar_test) if label == 1)
    selected_images = {"cat": cifar_test[cat_index][0], "dog": cifar_test[dog_index][0]}
    best_gamma = fitnets_df.sort_values("accuracy", ascending=False).iloc[0]["gamma"]
    best_fitnets_student = fitnets_visuals[best_gamma]["student"]
    part2_runs["Student + FitNets"] = {"student": best_fitnets_student, "history": None}

    heatmaps = []
    titles = []
    for class_name, image_tensor in selected_images.items():
        teacher_hook = FeatureHook(vgg_teacher.features[8])
        student_hook = FeatureHook(best_fitnets_student.pool1)
        _ = vgg_teacher(image_tensor.unsqueeze(0).to(device))
        teacher_map = mean_heatmap(teacher_hook.output)
        teacher_hook.close()
        _ = best_fitnets_student(image_tensor.unsqueeze(0).to(device))
        student_map = mean_heatmap(student_hook.output)
        student_hook.close()
        heatmaps.extend([teacher_map, student_map])
        titles.extend([f"Teacher - {class_name}", f"Student FitNets - {class_name}"])
    plot_heatmap_grid(heatmaps, titles, filename="part2_fitnets_feature_maps.png")
    plt.show()
else:
    print("FitNets pipeline prepared. Enable RUN_LONG_EXPERIMENTS after training or loading the CIFAR teacher.")


### P2.5 - RKD

We use the combined relational objective

\[
\mathcal{L}_{RKD} = \mathcal{L}_{KD} + \lambda_D \mathcal{L}_{RKD-D} + \lambda_A \mathcal{L}_{RKD-A}
\]

with the three requested settings: distance only, angle only, and the combined objective.


In [ ]:
if RUN_LONG_EXPERIMENTS:
    if not vgg_teacher_ready:
        raise RuntimeError("RKD requires a trained VGG-16 teacher checkpoint.")

    rkd_rows = []
    configs = [
        {"label": "RKD-D only", "lambda_D": 1.0, "lambda_A": 0.0},
        {"label": "RKD-A only", "lambda_D": 0.0, "lambda_A": 2.0},
        {"label": "RKD combined", "lambda_D": 1.0, "lambda_A": 2.0},
    ]

    for config in configs:
        student = TinyCNN(num_classes=part2_num_classes).to(device)
        optimizer = optim.Adam(student.parameters(), lr=1e-3)
        teacher_hook = FeatureHook(vgg_teacher.classifier[4])
        student_hook = FeatureHook(student.avgpool)

        def extra_loss_fn(base_loss, student_logits, teacher_logits):
            return base_loss + config["lambda_D"] * rkd_distance_loss(teacher_hook.output, student_hook.output) + config["lambda_A"] * rkd_angle_loss(teacher_hook.output, student_hook.output)

        history = fit_model(
            student=student,
            train_loader=cifar_train_loader,
            test_loader=cifar_test_loader,
            device=device,
            epochs=30,
            optimizer=optimizer,
            teacher=vgg_teacher,
            T=4.0,
            alpha=0.4,
            extra_loss_fn=extra_loss_fn,
        )
        part2_runs[config["label"]] = {"student": student, "history": history}
        rkd_rows.append(
            {
                "configuration": config["label"],
                "lambda_D": config["lambda_D"],
                "lambda_A": config["lambda_A"],
                "accuracy": history["test_acc"].iloc[-1],
            }
        )
        teacher_hook.close()
        student_hook.close()

    rkd_df = pd.DataFrame(rkd_rows)
    save_dataframe(rkd_df, "part2_rkd_results.csv")
    display(rkd_df)
else:
    print("RKD experiments prepared. They are intentionally gated behind RUN_LONG_EXPERIMENTS.")


### P2.6 - t-SNE on 200 test images

The statement asks for `50` test images per class. The helper subset below creates a balanced evaluation subset before extracting teacher and student representations.


In [ ]:
@torch.no_grad()
def collect_hook_features(model, module, loader, flatten: bool = True):
    hook = FeatureHook(module)
    features = []
    labels = []
    for inputs, target in loader:
        inputs = inputs.to(device)
        _ = model(adapt_inputs_to_model(model, inputs))
        feature_batch = hook.output
        if flatten:
            feature_batch = torch.flatten(feature_batch, 1)
        features.append(feature_batch.cpu())
        labels.append(target)
    hook.close()
    return torch.cat(features, dim=0), torch.cat(labels, dim=0)

balanced_indices = select_indices_by_label(cifar_test, per_label=50, num_classes=part2_num_classes)
balanced_cifar_loader = DataLoader(Subset(cifar_test, balanced_indices), batch_size=64, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

if RUN_LONG_EXPERIMENTS and "student_alone" in part2_runs and "RKD combined" in part2_runs:
    teacher_features, tsne_labels = collect_hook_features(vgg_teacher, vgg_teacher.classifier[4], balanced_cifar_loader)
    student_base_features, _ = collect_hook_features(part2_runs["student_alone"]["student"], part2_runs["student_alone"]["student"].avgpool, balanced_cifar_loader)
    student_rkd_features, _ = collect_hook_features(part2_runs["RKD combined"]["student"], part2_runs["RKD combined"]["student"].avgpool, balanced_cifar_loader)

    embeddings = {
        "Teacher": TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto").fit_transform(teacher_features.numpy()),
        "Student without distillation": TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto").fit_transform(student_base_features.numpy()),
        "Student with combined RKD": TSNE(n_components=2, random_state=SEED, init="pca", learning_rate="auto").fit_transform(student_rkd_features.numpy()),
    }
    plot_tsne_triptych(embeddings, tsne_labels.numpy(), cifar_class_names, filename="part2_tsne_triptych.png")
    plt.show()
else:
    print("t-SNE is ready but requires completed baseline and combined RKD runs.")


### P2.7 - Inter-class similarity matrices

Class prototypes are computed from the same balanced set of `50` test images per class. Cosine similarity matrices reveal which student best preserves the teacher geometry, especially for the semantically close pairs `cat/dog` and `deer/horse`.


In [ ]:
if RUN_LONG_EXPERIMENTS and "student_alone" in part2_runs and "RKD combined" in part2_runs:
    matrix_heatmaps = []
    matrix_titles = []

    teacher_features, teacher_labels = collect_hook_features(vgg_teacher, vgg_teacher.classifier[4], balanced_cifar_loader)
    teacher_proto = class_prototypes(teacher_features, teacher_labels, part2_num_classes)
    matrix_heatmaps.append(cosine_similarity_matrix(teacher_proto))
    matrix_titles.append("Teacher")

    for label, module_name in [
        ("student_alone", "Student alone"),
        ("Student + FitNets", "Student FitNets"),
        ("RKD combined", "Student RKD combined"),
    ]:
        if label not in part2_runs:
            print(f"{module_name} is not available yet.")
            continue
        model = part2_runs[label]["student"]
        student_features, student_labels = collect_hook_features(model, model.avgpool, balanced_cifar_loader)
        student_proto = class_prototypes(student_features, student_labels, part2_num_classes)
        matrix_heatmaps.append(cosine_similarity_matrix(student_proto))
        matrix_titles.append(module_name)

    plot_heatmap_grid(matrix_heatmaps, matrix_titles, cmap="viridis", filename="part2_similarity_matrices.png", annot=True)
    plt.show()
else:
    print("Similarity matrices are prepared. Run the corresponding Part 2 experiments first.")


### P2.8 and P2.9 - Final latency synthesis and analysis

Latency is measured on input shape `1 x 3 x 32 x 32` with `10` warm-up passes and `100` timed runs. The deployment column is a simple camera-oriented heuristic based on compactness and latency; keep the final report tied to the measured table.


In [ ]:
part2_synthesis_rows = [
    {
        "model": "Teacher (VGG16)",
        "size": format_size_kb(model_size_kb(vgg_teacher)),
        "latency_ms": measure_latency(vgg_teacher, input_shape=(1, 3, 32, 32), device=device),
        "precision": evaluate_macro_precision(vgg_teacher, cifar_test_loader) if vgg_teacher_ready else np.nan,
        "camera OK?": deployment_flag(model_size_kb(vgg_teacher), measure_latency(vgg_teacher, (1, 3, 32, 32), device), np.nan if not vgg_teacher_ready else 1.0, 5000.0, 25.0),
    }
]

for key, name in [
    ("student_alone", "Student alone"),
    ("student_kd", "Student + KD"),
    ("Student + FitNets", "Student + FitNets"),
    ("RKD combined", "Student + RKD combined"),
]:
    if key in part2_runs:
        model = part2_runs[key]["student"]
        precision = evaluate_macro_precision(model, cifar_test_loader)
        size_kb = model_size_kb(model)
        latency_ms = measure_latency(model, input_shape=(1, 3, 32, 32), device=device)
        part2_synthesis_rows.append(
            {
                "model": name,
                "size": format_size_kb(size_kb),
                "latency_ms": latency_ms,
                "precision": precision,
                "camera OK?": deployment_flag(size_kb, latency_ms, precision, 5000.0, 25.0),
            }
        )

part2_synthesis = pd.DataFrame(part2_synthesis_rows)
save_dataframe(part2_synthesis, "part2_final_synthesis.csv")
display(part2_synthesis)


The analysis to write after execution should cover the following points in about fifteen lines:

- **Response-based KD** is the cleanest baseline because it transfers class-level dark knowledge with almost no architectural changes.
- **FitNets** adds an explicit intermediate supervision signal, which is especially useful when the student struggles to build teacher-like feature maps early in the network.
- **RKD** does not try to match raw activations directly; instead, it preserves the relational structure between examples, which is why it is particularly interesting for the t-SNE plots and the class-prototype similarity matrices.
- In the final report, compare whether `cat/dog` and `deer/horse` remain close in the student embedding exactly as they do in the teacher embedding.
- If FitNets improves local feature heatmaps but RKD better preserves global geometry, say so explicitly instead of forcing a single narrative.
- The deployment choice for the camera should be based on the measured trade-off between precision, size, and latency, not on architecture prestige.
- If the combined RKD student is only marginally better but noticeably slower, the KD or FitNets student may still be the better deployment choice.
- If the camera budget is very tight, prefer the smallest student that preserves the teacher structure well enough for the target classes.


# 6. Final conclusion

This notebook now contains the full runnable TP pipeline:

- Part 1 implements filtered-MNIST response-based KD, temperature analysis, distillation regimes, and Attention Transfer.
- Part 2 scaffolds the complete filtered-CIFAR10 study with baseline KD, FitNets, RKD, t-SNE, similarity matrices, and deployment-oriented latency tables.

No experimental value is hard-coded. If the long runs are skipped, the notebook reports that the corresponding results are still pending instead of inventing numbers.
